In [20]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ファイルパス（先ほどリネームして保存した名前）
file_path = "../data/raw/cicids/02-14-2018.csv"

# 読み込み（400MB弱あるので、数秒かかります）
# low_memory=False を指定して型推論の警告を回避します
df = pd.read_csv(file_path, low_memory=False)

# カラム名にスペースが含まれている場合があるため、トリミングしておく
df.columns = df.columns.str.strip()

print(f"データの総行数: {len(df):,}")
print("-" * 30)
# 最初の5行を表示
display(df.head())

データの総行数: 1,048,575
------------------------------


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0,0,14/02/2018 08:31:01,112641719,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56320859.5,139.300036,56320958,56320761,Benign
1,0,0,14/02/2018 08:33:50,112641466,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56320733.0,114.551299,56320814,56320652,Benign
2,0,0,14/02/2018 08:36:39,112638623,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56319311.5,301.934596,56319525,56319098,Benign
3,22,6,14/02/2018 08:40:13,6453966,15,10,1239,2273,744,0,...,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign
4,22,6,14/02/2018 08:40:23,8804066,14,11,1143,2209,744,0,...,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign


In [21]:
# Label列が 'Benign' の行を探し、その 'Label' 列だけを '-' に書き換える
df.loc[df['Label'] == 'Benign', 'Label'] = '-'

# 確認
print("--- 変換後のラベル内訳 ---")
print(df['Label'].value_counts())

--- 変換後のラベル内訳 ---
Label
-                 667626
FTP-BruteForce    193360
SSH-Bruteforce    187589
Name: count, dtype: int64


In [ ]:
# --- ステップ1: 掃除 ---
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# --- ステップ2: ベクトルDB用の「知識」を抽出 (正常のみ) ---
normal_df = df[df['Label'] == '-']

print(f"知識データ(DB用): {len(normal_df)}件")
print(f"テストデータ合計: {len(df)}件")
print("テストデータ内の内訳:\n", df['Label'].value_counts())

知識データ（DB用）: 663808件
テストデータ合計: 1044751件
テストデータ内の内訳:
 Label
-                 663808
FTP-BruteForce    193354
SSH-Bruteforce    187589
Name: count, dtype: int64


In [23]:
# 厳選10カラム
selected_features = [
    'Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 
    'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'Flow Byts/s', 
    'Flow Pkts/s', 'Init Fwd Win Byts', 'Idle Mean'
]

def serialize_selected_logs(row):
    """
    選定した10列を結合してテキスト化する
    """
    # 数値が科学的表記(1e+05等)にならないよう、floatを整形
    parts = []
    for col in selected_features:
        val = row[col]
        # float型で整数に近い場合は整数表示、そうでない場合は小数点以下2桁程度にする
        if isinstance(val, float):
            val = f"{val:.2f}" if not val.is_integer() else f"{int(val)}"
        parts.append(f"{col}: {val}")
    
    return " | ".join(parts)

# データの読み込み（前回のCSV）

# 10列絞り込みバージョンのテキスト作成
df['message'] = df.apply(serialize_selected_logs, axis=1)
normal_df['message'] = normal_df.apply(serialize_selected_logs, axis=1)

# 文字数を確認
print(f"絞り込み後の1行あたりの平均文字数: {df['message'].str.len().mean():.1f} 文字")
print("\n--- 生成されたテキストの例 ---")
print(df['message'].iloc[0])

絞り込み後の1行あたりの平均文字数: 191.4 文字

--- 生成されたテキストの例 ---
Dst Port: 0 | Protocol: 0 | Flow Duration: 112641719 | Tot Fwd Pkts: 3 | Tot Bwd Pkts: 0 | TotLen Fwd Pkts: 0 | Flow Byts/s: 0 | Flow Pkts/s: 0.03 | Init Fwd Win Byts: -1 | Idle Mean: 56320859.50


In [26]:
# 2. 'Label' と 'text' 列だけを抽出した新しいデータフレームを作成
test_df = df[['Label', 'message']].copy()
test_df.columns = ['label', 'message']
normal_all_df = normal_df[['Label', 'message']].copy()
normal_all_df.columns = ['label', 'message']

In [31]:
test_df.to_csv("../data/processed/cicids/cicids_test.csv", index=False)
normal_all_df.to_csv("../data/processed/cicids/cicids_normal_all.csv", index=False)